# **RSM434 ALGO2e: Multi-Security Market Maker**

**Where the money is.** ANON market orders pay the spread to whoever is at the front of the queue, and every passive fill earns or costs the passive rebate. In ticks of $0.01:

| | passive (make) | active (take) | passive round trip |
|---|---|---|---|
| CNR | +0.23 | −0.27 | spread + 0.46 |
| AC  | +0.11 | −0.15 | spread + 0.22 |
| RY  | −0.20 | +0.14 | spread − 0.40 |

These are the case brief's numbers. Once the case is live, the algo reads the fee and rebate from the server and quotes with those. RIT charges the fee on every fill and pays its listed rebate on top for passive ones, so the server's CNR rebate of 0.50 nets to the +0.23 above.

All three fall out of a single quote formula. The **reservation price** is the microprice minus an inventory skew. A bid at price *p* has an expected edge of `res − p + rebate`, an ask `p − res + rebate`. Each side is quoted at the most aggressive price (penny or join) whose edge clears a floor. Gaps of 3+ ticks get pennied: 0.5 tick of spread plus the rebate, which RY clears only just. A 2-tick gap gets pennied only when the microprice leans our way, since the penny lands on the mid and leaves just the rebate. RY is also the cheapest to dump, because taking liquidity there *pays*.

**1) Assume we are the fastest: FAST mode**
* Penny the best price from anyone else by one tick whenever the edge clears a 0.3-tick floor. The first order at a new best price gets the next ANON order, and being fastest means we win that race and every re-penny after it.
* Put full size at the front (5,000 shares), plus a second order 2 ticks deeper that fills at a much better price when a large order sweeps several levels.
* Never chase a 1-tick move. When ANON opens a new price level it is already ahead of us there, so moving just gives up our queue priority. A rival's penny moves our target 2 ticks, so we still answer it at once.

**2) If we are not the fastest: SLOW mode, chosen per ticker automatically**
* *Detect.* Each cycle, on each side where we have an order and a rival quotes a price we would also quote, check who holds the front. A rival at a better price, or at the same price with a lower `order_id`, counts as a loss. A side where we have no order, or where the rival sits past our edge floor, is not a race and is not scored. An EMA of losses above 0.5 switches to SLOW, below 0.3 switches back. Every 15 ticks we retry FAST in case the rival has left.
* *Adapt.* On a side whose best price is a rival's, stop the penny war and join it instead: a faster rival wins every re-penny, so pennying only burns edge. Require a 0.4-tick edge, because our quotes go stale for longer. Shift size deeper (3,000 at the front, 5,000 two ticks back), because big orders that sweep several levels are where a slower algo still gets paid. A side whose best price is ANON's is still pennied with FAST settings: there is no one to race there, and joining would put us at the back of ANON's queue, where only the sweeps that run the price against us reach us.

**Always on**
* Risk: an 8,000-share cap per ticker, so all three maxed together stays inside 25,000 gross and net. The skew is 2 ticks at a full cap. When the skewed edge still clears the taker fee by 0.5 tick, cross the spread to shed inventory. A sweep can still fill old and new quotes together during a reprice, so anything past the cap is bought or sold back at market straight away: the limit fine is $1 a share, against a cent or two to cross. Setting `STRICT_ROOM` makes orders still being cancelled count against the cap until the cancel is confirmed, which prevents the overfill at the cost of one round trip per reprice.
* Execution: each cycle sends 5 parallel GETs (the case itself only every 0.2 s), decides, then sends cancels and orders in parallel. Orders that depend on a cancel go out only once it is confirmed. Only orders whose price or size actually changed are touched, which keeps queue priority. On an HTTP 429 the loop waits as long as RIT asks, once, then returns to full speed.
* Close: for the last 8 ticks, only quote orders that reduce inventory. In the last 2 ticks, flatten at market.

# **Settings**

In [ ]:
import base64
import math
import time
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import requests
from requests.adapters import HTTPAdapter

BASE_URL = 'http://flserver.rotman.utoronto.ca:16585/v1'
USERNAME = '1010264928'
PASSWORD = 'fa'

# ---- case constants (case brief). FEE and REBATE are replaced by the server's values once the case is live.
TICKERS = ('CNR', 'RY', 'AC')
FEE    = {'CNR': 0.0027, 'RY': -0.0014, 'AC': 0.0015}   # per share, charged on every fill, passive or active
REBATE = {'CNR': 0.0023, 'RY': -0.0020, 'AC': 0.0011}   # net per passive share: the server's limit_order_rebate minus FEE
BRIEF  = {t: (FEE[t], REBATE[t]) for t in TICKERS}
TICK, MAXQ, LIMIT = 0.01, 5000, 25000

# ---- strategy (prices and edges in ticks)
CAP       = int(0.96 * LIMIT / len(TICKERS))  # per-ticker inventory cap; all caps together stay inside LIMIT
SKEW      = 2.0    # reservation-price shift at a full cap
TAKE_EDGE = 0.5    # edge left after the taker fee before we cross the spread to shed inventory
TOPUP     = 1000   # resize a resting order only when this far off target
WIND      = 8      # final ticks: quote only to reduce inventory
CLOSE     = 2      # final ticks: flatten at market
STICKY    = 1      # ticks a resting quote may sit behind its target before we chase (keeps queue priority)
DEEP      = 2      # second order sits this many ticks behind the front one; must exceed STICKY
PROBE     = 15     # every PROBE ticks, retry FAST in case rivals have left
ALPHA, TO_SLOW, TO_FAST = 0.02, 0.5, 0.3      # race-loss EMA and its hysteresis band
STRICT_ROOM = False  # True: orders still being cancelled count against CAP until confirmed (no overfill, but a reprice waits a round trip)
MODES = {  # edge: min edge per fill | penny: ticks inside the best price | front: front size, the rest goes deep
    'FAST': dict(edge=0.3, penny=1, front=MAXQ),
    'SLOW': dict(edge=0.4, penny=0, front=3000),
}
PACE       = 0.01  # fastest cycle in seconds; an HTTP 429 adds only the wait RIT asks for
BOOK_DEPTH = 20
TIMEOUT    = 0.5   # seconds per trading request; a slow one is dropped and retried next cycle
CASE_EVERY = 0.2   # seconds between /case reads; only the tick and status come from it

# ---- connection: one keep-alive session shared by a thread pool
s = requests.Session()
s.headers['Authorization'] = 'Basic ' + base64.b64encode(f'{USERNAME}:{PASSWORD}'.encode()).decode()
s.mount('http://', HTTPAdapter(pool_maxsize=32))
POOL = ThreadPoolExecutor(32)
WAIT = [0.0]       # longest Retry-After seen this cycle

# Diagnostics stay in memory during the case. report() reads the server only after it stops.
STATS  = {'http429': 0, 'cycles': 0, 'miss': 0, 'sent': 0, 'cancels': 0, 'held': 0}
NET    = defaultdict(lambda: {'ms': [], '429': 0, 'err': 0})   # per endpoint: latency, rate limits, failures
WORK   = []          # work time of every cycle, ms
TRACE  = []          # one row per tick, taken from the snapshot the loop already fetched
RACES  = {}          # per ticker: [races won, races lost] as scored by lost()
RIVALS = set()       # non-ANON trader ids seen in the books
SPAN   = [0.0, 0.0]  # perf_counter at the first and the last live cycle

# **API**

In [ ]:
def api(method, path, timeout=TIMEOUT, **params):
    """One request -> parsed JSON, or None on any failure (the loop just retries next cycle)."""
    net = NET[f'{method} {path}']
    t0 = time.perf_counter()
    try:
        r = s.request(method, f'{BASE_URL}/{path}', params=params, timeout=timeout)
        if r.status_code == 429:
            WAIT[0] = max(WAIT[0], float(r.headers.get('Retry-After') or r.json().get('wait', 0.05)))
            STATS['http429'] += 1
            net['429'] += 1
            return None
        if r.ok:
            return r.json()
        net['err'] += 1
        return None
    except (requests.RequestException, ValueError):
        net['err'] += 1
        return None
    finally:
        net['ms'].append((time.perf_counter() - t0) * 1000)


def snapshot(with_case):
    """Positions, our open orders and all three books, plus the case when asked, fetched in parallel."""
    reqs = {'sec': ('securities', {}), 'orders': ('orders', {}),
            **{t: ('securities/book', {'ticker': t, 'limit': BOOK_DEPTH}) for t in TICKERS}}
    if with_case:
        reqs['case'] = ('case', {})
    futs = {k: POOL.submit(api, 'GET', path, **q) for k, (path, q) in reqs.items()}
    return {k: f.result() for k, f in futs.items()}


def send(t, action, p, q):
    """p is in ticks; p=None sends a market order."""
    if p is None:
        return api('POST', 'orders', ticker=t, type='MARKET', quantity=q, action=action)
    return api('POST', 'orders', ticker=t, type='LIMIT', quantity=q, action=action, price=round(p * TICK, 2))


def execute(cancel, now, later, need):
    """Cancels and safe orders go out together. The rest go out once every cancel on their ticker is confirmed;
    an unconfirmed cancel means that order filled, so its ticker waits for the next snapshot instead."""
    jobs = [POOL.submit(send, *o) for o in now]
    done = set()
    if cancel:
        out = api('POST', 'commands/cancel', ids=','.join(map(str, cancel)))
        done = set((out or {}).get('cancelled_order_ids') or [])
    ready = [o for o in later if need[o[0]] <= done]
    jobs += [POOL.submit(send, *o) for o in ready]
    STATS['sent'] += len(now) + len(ready)
    STATS['cancels'] += len(cancel)
    STATS['held'] += len(later) - len(ready)
    for j in jobs:
        j.result()

# **Strategy**

In [ ]:
P   = lambda o: round(o['price'] / TICK)             # price -> integer ticks
rem = lambda o: o['quantity'] - o['quantity_filled']
ME, LAG, MODE = None, {}, {}


def load_fees(case, sec):
    """Quote with the server's fee and net passive rebate, and report where they differ from the case brief.
    RIT charges the fee on every fill and pays limit_order_rebate on top for passive ones, so the net
    passive rebate is the difference of the two."""
    by = {d['ticker']: d for d in sec}
    lines = [f"{case.get('name', '?')} is live. Per share, server (case brief):"]
    for t in TICKERS:
        d = by.get(t, {})
        fee, reb = BRIEF[t]
        if d.get('trading_fee') is None or d.get('limit_order_rebate') is None:
            lines.append(f'  {t:4} not reported by the server, quoting with the case brief')
            continue
        FEE[t], REBATE[t] = d['trading_fee'], d['limit_order_rebate'] - d['trading_fee']
        differ = abs(FEE[t] - fee) > 1e-9 or abs(REBATE[t] - reb) > 1e-9
        lines.append(f"  {t:4} fee {FEE[t]:+.4f} ({fee:+.4f})   net passive rebate {REBATE[t]:+.4f} ({reb:+.4f})"
                     + ('   DIFFERENT from the brief, quoting with the server values' if differ else ''))
    return '\n'.join(lines)


def lost(side, lim, d):
    """1 if a rival holds the front at a price we would quote, 0 if we do, None if there was no race.
    lim is our most aggressive acceptable price; d is +1 for bids, -1 for asks."""
    mine = next((o for o in side if o['trader_id'] == ME), None)
    rival = next((o for o in side if o['trader_id'] not in (ME, 'ANON') and d * (lim - P(o)) >= 0), None)
    if mine is None or rival is None:
        return None
    return int(d * (P(rival) - P(mine)) > 0 or P(rival) == P(mine) and rival['order_id'] < mine['order_id'])


def reconcile(want, mine):
    """Diff target quotes (action, price, qty, band) against resting orders, touching as few as possible."""
    rest = {}
    for o in mine:
        g = rest.setdefault((o['action'], P(o)), [0, []])
        g[0] += rem(o)
        g[1].append(o['order_id'])
    cancel, post = [], []
    for a, p, q, band in want:
        d = 1 if a == 'BUY' else -1                  # a resting order may sit up to `band` ticks less aggressive
        key = next((k for k in ((a, p - d * i) for i in range(band + 1)) if k in rest), None)
        have, ids = rest.pop(key) if key else (0, [])
        if key and (not q or have > q + TOPUP):      # unwanted or oversized: replace
            cancel += ids
            key, have = None, 0
        if q > have + (TOPUP if key else 0):         # missing or under-filled: add (keeps the rest's priority)
            post.append((a, key[1] if key else p, q - have))
    for _, ids in rest.values():
        cancel += ids
    return cancel, post


def plan(t, book, mine, pos, tick, T):
    """One ticker -> (ids to cancel, orders to send now, orders to send once those cancels are confirmed)."""
    if tick >= T - CLOSE:                            # last seconds: flatten at market
        flat = [('SELL' if pos > 0 else 'BUY', None, min(abs(pos), MAXQ))] if pos else []
        return [o['order_id'] for o in mine], [], flat

    bids = [o for o in book['bids'] if o['trader_id'] != ME]
    asks = [o for o in book['asks'] if o['trader_id'] != ME]
    if not bids or not asks:
        return [], [], []
    bb, ba = P(bids[0]), P(asks[0])
    bq = sum(rem(o) for o in bids if P(o) == bb)
    aq = sum(rem(o) for o in asks if P(o) == ba)
    r, f = REBATE[t] / TICK, FEE[t] / TICK
    res = (bb * aq + ba * bq) / (bq + aq) - SKEW * pos / CAP      # microprice shifted against inventory

    edge = MODES[MODE[t]]['edge']                    # are we the fastest on this ticker?
    for side, lim, d in ((book['bids'], math.floor(res + r - edge), 1), (book['asks'], math.ceil(res - r + edge), -1)):
        x = lost(side, lim, d)
        if x is not None:
            LAG[t] += ALPHA * (x - LAG[t])
            RACES[t][x] += 1
    RIVALS.update(o['trader_id'] for o in book['bids'] + book['asks'] if o['trader_id'] not in (ME, 'ANON'))
    MODE[t] = 'SLOW' if LAG[t] > TO_SLOW else 'FAST' if LAG[t] < TO_FAST else MODE[t]
    # SLOW only stops re-pennying a rival; a side whose best price is ANON's is pennied regardless.
    mb = MODES['FAST' if bids[0]['trader_id'] == 'ANON' else MODE[t]]
    ma = MODES['FAST' if asks[0]['trader_id'] == 'ANON' else MODE[t]]

    take = []
    if abs(pos) > CAP:                               # overfilled: the limit fine is $1/share, so shed the excess at market
        take.append(('SELL' if pos > 0 else 'BUY', None, min(abs(pos) - CAP, MAXQ)))
    else:                                            # cross to shed inventory once the skew pays the taker fee
        for a, p, e, size, inv in (('SELL', bb, bb - res - f, bq, pos), ('BUY', ba, res - ba - f, aq, -pos)):
            if inv > 0 and e > TAKE_EDGE:
                take.append((a, p, int(min(inv, size, MAXQ, math.ceil((e - TAKE_EDGE) * CAP / SKEW)))))

    bid = min(bb + mb['penny'], ba - 1, math.floor(res + r - mb['edge']))   # most aggressive price that clears the edge
    ask = max(ba - ma['penny'], bb + 1, math.ceil(res - r + ma['edge']))
    if bid >= ask:                                   # 2-tick spread: only one side can penny
        bid, ask = (bid, bid + 1) if res > (bb + ba) / 2 else (ask - 1, ask)
    rb, ra = (max(0, -pos), max(0, pos)) if tick >= T - WIND else (CAP - pos, CAP + pos)
    want = []
    for a, p, room, d, m in (('BUY', bid, rb, -1, mb), ('SELL', ask, ra, 1, ma)):
        q0 = max(0, min(room, m['front']))
        want += [(a, p, q0, STICKY), (a, p + d * DEEP, max(0, min(room - q0, MAXQ)), STICKY)]
    cancel, post = reconcile(want, mine)

    hi_bid = max((P(o) for o in mine if o['action'] == 'BUY'), default=-math.inf)
    lo_ask = min((P(o) for o in mine if o['action'] == 'SELL'), default=math.inf)

    def crosses(o):                                  # would trade with ourselves; a market order crosses any order of ours
        if o[0] == 'BUY':
            return lo_ask < math.inf if o[1] is None else o[1] >= lo_ask
        return hi_bid > -math.inf if o[1] is None else o[1] <= hi_bid

    free = {'BUY': rb, 'SELL': ra}                   # room left if every resting order fills, cancelled ones included
    for o in mine:
        free[o['action']] -= rem(o)
    now, later = [], []
    for sheds, group in ((True, take), (False, post)):
        for o in group:
            safe = not crosses(o) and (sheds or not STRICT_ROOM or o[2] <= free[o[0]])
            (now if safe else later).append(o)
            free[o[0]] -= o[2]
    return cancel, now, later

# **Main loop**

In [ ]:
def main():
    global ME
    STATS.update(http429=0, cycles=0, miss=0, sent=0, cancels=0, held=0)
    for log in (NET, WORK, TRACE, RIVALS):
        log.clear()
    me = api('GET', 'trader', timeout=5)
    if not me:
        print('Cannot reach RIT: check BASE_URL, USERNAME and PASSWORD.')
        return
    ME = me['trader_id']
    LAG.update(dict.fromkeys(TICKERS, 0.0))
    MODE.update(dict.fromkeys(TICKERS, 'FAST'))
    RACES.update({t: [0, 0] for t in TICKERS})
    live, last, case, read = False, -1, None, -math.inf
    print(f'{ME} connected, waiting for the case.')
    try:
        while True:
            t0 = time.perf_counter()
            snap = snapshot(t0 - read >= CASE_EVERY)
            fresh = snap.pop('case', None)
            if fresh:
                case, read = fresh, t0
            if case and case['status'] != 'ACTIVE':
                if live and case['status'] == 'STOPPED':
                    break
                time.sleep(0.2)
                continue
            if case and all(v is not None for v in snap.values()):
                if not live:
                    live = True
                    SPAN[0] = t0
                    print(load_fees(case, snap['sec']), flush=True)
                STATS['cycles'] += 1
                tick, T = case['tick'], case['ticks_per_period']
                pos = {d['ticker']: int(d['position']) for d in snap['sec']}
                cancel, now, later, need = [], [], [], {}
                for t in TICKERS:
                    c, n, l = plan(t, snap[t], [o for o in snap['orders'] if o['ticker'] == t], pos[t], tick, T)
                    cancel += c
                    need[t] = set(c)
                    now += [(t, *o) for o in n]
                    later += [(t, *o) for o in l]
                execute(cancel, now, later, need)
                SPAN[1] = time.perf_counter()
                work = (SPAN[1] - t0) * 1000
                WORK.append(work)
                if tick != last:
                    last = tick
                    try:
                        sample(tick, snap, work)
                    except Exception:
                        pass
                    if tick % PROBE == 0:
                        LAG.update(dict.fromkeys(TICKERS, 0.0))
            else:
                STATS['miss'] += 1
            wait, WAIT[0] = WAIT[0], 0.0
            time.sleep(max(wait, PACE - (time.perf_counter() - t0)))
    finally:
        api('POST', 'commands/cancel', timeout=5, all=1)
        print('Stopped, all orders cancelled.', flush=True)
        try:
            report()
        except Exception as exc:
            print('Report failed:', exc, flush=True)

# **Diagnostics**

Nothing here sends requests during a trading cycle. The loop only records timings, counters and a few numbers from the snapshot it already fetched. `report()` is read-only. It runs once the case stops, and again from the last cell if you want the same recap back. RIT deletes fills and transactions when the next case starts, so the saved file under `diagnostics/` is the copy that remains.

In [ ]:
import os
from datetime import datetime


def fetch(path, **params):
    """GET with a few retries. Used by the report only, never by the trading loop."""
    for _ in range(4):
        out = api('GET', path, timeout=5, **params)
        if out is not None:
            return out
        time.sleep(max(WAIT[0], 0.05))
    return None


def sample(tick, snap, work_ms):
    """One row per tick from the snapshot already in hand. No extra requests."""
    by = {d['ticker']: d for d in snap['sec']}
    row, pnl = [], 0.0
    for t in TICKERS:
        d = by.get(t)
        if d is None:
            return
        pos, realized, unrealized = int(d['position']), float(d['realized']), float(d['unrealized'])
        row.append((pos, realized, unrealized, MODE[t]))
        pnl += realized + unrealized
    TRACE.append((tick, work_ms, tuple(row)))
    if tick % 10 == 0:
        print(
            f"tick {tick:3d} | pnl {pnl:+9.2f} | "
            + '  '.join(f"{t} {row[i][0]:+6d} {row[i][3]}" for i, t in enumerate(TICKERS))
            + f" | {work_ms:.0f} ms",
            flush=True,
        )


def _pct(xs, p):
    xs = sorted(xs)
    return xs[min(len(xs) - 1, int(round((len(xs) - 1) * p)))]


def report():
    """Read-only recap of P&L, fills, fees, limits and speed. Does not send orders."""
    stats = dict(STATS)
    net = {k: {'ms': list(v['ms']), '429': v['429'], 'err': v['err']} for k, v in NET.items()}
    jobs = {
        'case': POOL.submit(fetch, 'case'),
        'trader': POOL.submit(fetch, 'trader'),
        'sec': POOL.submit(fetch, 'securities'),
        'limits': POOL.submit(fetch, 'limits'),
        'filled': POOL.submit(fetch, 'orders', status='TRANSACTED'),
        'tx': POOL.submit(fetch, 'transactions', limit=100000),
    }
    data = {k: fut.result() for k, fut in jobs.items()}
    case, trader = data['case'] or {}, data['trader'] or {}
    sec, limits = data['sec'] or [], data['limits'] or []
    filled, tx = data['filled'] or [], data['tx'] or []
    lines = []
    add = lines.append

    add(f"{case.get('name', '?')}   period {case.get('period', '?')}   "
        f"tick {case.get('tick', '?')}/{case.get('ticks_per_period', '?')}   {case.get('status', '?')}")
    if case:
        add('trading limits: ' + ('hard reject' if case.get('is_enforce_trading_limits') else 'a fine, not a hard reject'))
    add(f"trader {trader.get('trader_id', '?')}    NLV {trader.get('nlv', 0):,.2f}    fines {trader.get('total_fines', 0):,.2f}")
    add('')
    add(f"{'':6} {'pos':>8} {'vwap':>8} {'last':>8} {'realized':>10} {'unrealized':>11} {'volume':>10}"
        f"   fee      rebate   net passive (brief)")
    by = {d['ticker']: d for d in sec}
    for t in TICKERS:
        d = by.get(t)
        if not d:
            add(f'{t}  not in /securities')
            continue
        fee, reb = d.get('trading_fee') or 0, d.get('limit_order_rebate') or 0
        differ = abs(fee - BRIEF[t][0]) > 1e-9 or abs(reb - fee - BRIEF[t][1]) > 1e-9
        add(f"{t:6} {d.get('position', 0):8.0f} {d.get('vwap') or 0:8.2f} {d.get('last') or 0:8.2f} "
            f"{d.get('realized') or 0:10.2f} {d.get('unrealized') or 0:11.2f} {d.get('volume') or 0:10.0f}   "
            f"{fee:+.4f}  {reb:+.4f}  {reb - fee:+.4f} ({BRIEF[t][1]:+.4f})"
            + ('   DIFFERENT from the brief' if differ else ''))
        notes = []
        if d.get('max_trade_size') and int(d['max_trade_size']) != MAXQ:
            notes.append(f"server max order {d['max_trade_size']:.0f} shares; this file caps a quote at {MAXQ}")
        if d.get('api_orders_per_second'):
            notes.append(f"server allows {int(d['api_orders_per_second'])} orders/s")
        if d.get('execution_delay_ms'):
            notes.append(f"execution delay {int(d['execution_delay_ms'])} ms")
        if notes:
            add('       ' + '; '.join(notes))
    add('')
    add('limits')
    if not limits:
        add('  none returned')
    for lim in limits:
        add(f"  {lim.get('name')}  gross {lim.get('gross', 0):.0f}/{lim.get('gross_limit', 0):.0f}"
            f"   net {lim.get('net', 0):.0f}/{lim.get('net_limit', 0):.0f}"
            f"   fine {lim.get('gross_fine', 0):.2f}/sh gross, {lim.get('net_fine', 0):.2f}/sh net")

    add('')
    add(f'our fills  ({len(filled)} transacted orders)')
    agg = defaultdict(lambda: {'n': 0, 'sh': 0.0, 'buy': 0.0, 'sell': 0.0, 'mkt': 0.0, 'lmt': 0.0})
    for o in filled:
        g = agg[o.get('ticker')]
        sh = float(o.get('quantity_filled') or 0)
        g['n'] += 1
        g['sh'] += sh
        g['buy' if o.get('action') == 'BUY' else 'sell'] += sh
        g['mkt' if o.get('type') == 'MARKET' else 'lmt'] += sh
    if not agg:
        add('  none. The server clears this log when the next case starts.')
    else:
        add(f"  {'':6} {'orders':>7} {'shares':>10} {'bought':>10} {'sold':>10} {'market':>10} {'limit':>10}")
        for t in list(TICKERS) + [t for t in agg if t not in TICKERS]:
            g = agg.get(t)
            if not g:
                continue
            add(f"  {t:6} {g['n']:7d} {g['sh']:10.0f} {g['buy']:10.0f} {g['sell']:10.0f} {g['mkt']:10.0f} {g['lmt']:10.0f}")

    add('')
    add(f'transaction log  ({len(tx)} rows)')
    cash, ntx = defaultdict(float), defaultdict(int)
    for row in tx:
        key = (row.get('ticker') or '-', row.get('type') or '?')
        cash[key] += float(row.get('cash_flow') or 0)
        ntx[key] += 1
    if not cash:
        add('  none. The server clears this log when the next case starts.')
    else:
        prefer = ('ORDER', 'COMMISSION', 'LIMIT_REBATE', 'MARK_TO_MARKET', 'CLOSE_OUT',
                  'GROSS_LIMIT_FINE', 'NET_LIMIT_FINE', 'CASE_FINE')
        add(f"  {'ticker':6} {'type':18} {'n':>6} {'cash':>12}")
        for ticker, kind in sorted(cash, key=lambda k: (k[0], prefer.index(k[1]) if k[1] in prefer else 99, k[1])):
            add(f"  {ticker:6} {kind:18} {ntx[(ticker, kind)]:6d} {cash[(ticker, kind)]:12.2f}")
        add(f"  {'':6} {'ALL':18} {sum(ntx.values()):6d} {sum(cash.values()):12.2f}")

    add('')
    add('this run, from the in-memory trace (no extra requests)')
    if not TRACE:
        add('  empty. This notebook was not the process that traded, or the case had not ticked yet.')
    else:
        first = [r[1] for r in TRACE]
        add(f"  ticks {TRACE[0][0]}\u2013{TRACE[-1][0]}   samples {len(TRACE)}")
        add(f"  cycle work, every cycle      median {_pct(WORK, 0.5):.0f} ms   p95 {_pct(WORK, 0.95):.0f} ms   max {max(WORK):.0f} ms")
        add(f"  cycle work, first of a tick  median {_pct(first, 0.5):.0f} ms   p95 {_pct(first, 0.95):.0f} ms   max {max(first):.0f} ms")
        add(f"  {stats['cycles'] / max(SPAN[1] - SPAN[0], 1e-9):.1f} cycles/s   snapshots {stats['cycles']}   missed {stats['miss']}"
            f"   HTTP 429 {stats['http429']}   orders sent {stats['sent']}   cancels {stats['cancels']}"
            f"   held for an unconfirmed cancel {stats['held']}")
        add(f"  {'':6} {'max long':>9} {'max short':>10} {'end pos':>8} {'FAST':>6} {'SLOW':>6}")
        for i, t in enumerate(TICKERS):
            poss = [r[2][i][0] for r in TRACE]
            modes = [r[2][i][3] for r in TRACE]
            add(f"  {t:6} {max(poss):9d} {min(poss):10d} {poss[-1]:8d} {modes.count('FAST'):6d} {modes.count('SLOW'):6d}")
        pnls = [sum(r[2][i][1] + r[2][i][2] for i in range(len(TICKERS))) for r in TRACE]
        add(f"  pnl min {min(pnls):+.2f}   max {max(pnls):+.2f}   last {pnls[-1]:+.2f}")
        add('  path (realized + unrealized)')
        shown = [r for r in TRACE if r[0] % 30 == 0]
        if not shown or shown[-1] is not TRACE[-1]:
            shown.append(TRACE[-1])
        for r in shown:
            pnl = sum(r[2][i][1] + r[2][i][2] for i in range(len(TICKERS)))
            poss = '  '.join(f"{TICKERS[i]} {r[2][i][0]:+d} {r[2][i][3]}" for i in range(len(TICKERS)))
            add(f"    tick {r[0]:3d}  {pnl:+10.2f}   {poss}")

    add('')
    add('requests')
    add(f"  {'':22} {'n':>7} {'p50 ms':>7} {'p95 ms':>7} {'max ms':>7} {'429':>5} {'failed':>7}")
    for k, v in sorted(net.items()):
        if v['ms']:
            add(f"  {k:22} {len(v['ms']):7d} {_pct(v['ms'], 0.5):7.0f} {_pct(v['ms'], 0.95):7.0f} "
                f"{max(v['ms']):7.0f} {v['429']:5d} {v['err']:7d}")

    add('')
    add('races (a side where we and a rival both quoted a price we would take)')
    for t in TICKERS:
        won, lose = RACES.get(t, [0, 0])
        add(f"  {t:6} scored {won + lose:6d}   lost {lose / max(won + lose, 1):5.0%}")
    add('  rivals seen: ' + (', '.join(sorted(RIVALS)) if RIVALS else 'none'))

    text = '\n'.join(lines)
    print('\n' + text, flush=True)
    os.makedirs('diagnostics', exist_ok=True)
    path = os.path.join('diagnostics', datetime.now().strftime('%Y%m%d-%H%M%S') + '.txt')
    latest = os.path.join('diagnostics', 'latest.txt')
    for name in (path, latest):
        with open(name, 'w', encoding='utf-8') as f:
            f.write(text + '\n')
    print(f'\nSaved {path}', flush=True)
    return text


# **Run**

In [ ]:
try:
    main()
except KeyboardInterrupt:
    pass

1010264928 connected, waiting for the case.
ALGO2e v2 is live. Server fee/rebate vs the constants in this file:
  CNR  fee +0.0027 (file +0.0027)   rebate +0.0050 (file +0.0023)   DIFFERENT — quotes use the file, not the server
  RY  fee -0.0014 (file -0.0014)   rebate -0.0034 (file -0.0020)   DIFFERENT — quotes use the file, not the server
  AC  fee +0.0015 (file +0.0015)   rebate +0.0026 (file +0.0011)   DIFFERENT — quotes use the file, not the server
tick  70 | pnl  -1032.61 | CNR  -2500 FAST  RY  -3000 FAST  AC   -936 FAST | 52 ms


In [ ]:
# Read-only. Run this cell by itself to pull the server log again.
# RIT deletes fills and transactions when the next case starts.
report()
